# Flux post-processing chain: IRGA72 input, Level-1 fluxes and meteo (2016-2026.09)

**notebook version**: `3` (25 Oct 2026)

## ℹ️ About this notebook

Joins the merged **Level-1** fluxes of the **IRGA72** gas analyser (`20_L1_FLUXES/IRGA72/99`) with the merged meteo
products (`10_METEO/30_PRODUCTS/99`) on one 30MIN index, and stores the result in the external
data folder as parquet. The flux notebooks of this folder (`01`, `11`, `15`) read this file:
the gap-filling of Level 4.1 takes its drivers from the meteo columns.

No value is computed or corrected here. The flux columns are exactly what `20_L1_FLUXES/IRGA72/99` wrote and the meteo
columns exactly what `99` wrote. The joined record runs from `PERIOD_START`, 1 January 2016 00:15,
to 30 September 2026. The flux columns are empty before the first `IRGA72` half-hour on 11 January
2016; the meteo columns hold values from the first record on.

**This file has to be rebuilt whenever `99` changes.** `99` currently merges the meteo products
that are extended to September 2026; every column it adds later reaches this file only through a
new run of this notebook.

***

## 📇 Data sources

- `20_L1_FLUXES/IRGA72/99_IRGA72_L1_MERGED_2016-2026.09.parquet`: the Level-1 fluxes, one
  record per half-hour, each record taken whole from one EddyPro run.
- `10_METEO/30_PRODUCTS/99_METEO_MERGED_2004-2026.09.parquet`: every merged meteo series with its
  provenance flags. Its own information file, `99_METEO_MERGED_2004-2026.09_INFO.md`, describes
  the columns and flag codes.

Both live in the external data folder and are addressed by absolute path. Both are indexed on
`TIMESTAMP_MIDDLE`, local standard time (UTC+01:00), so they are joined without any shift.

**The meteo columns are taken whole.** All series of `99` arrive, the measured and the homogenised
columns side by side, so the choice of driver stays with the chain. The meteo names end in
`_T1_47_1` and the EddyPro names never do, which the checks below assert. The EddyPro biomet
columns were already dropped in `20_L1_FLUXES/IRGA72/99`.

**Two driver names changed against the 2016-2024 version of this file.** It was built on an older
meteo file. `TA_T1_47_1_gfXG` and `SW_IN_T1_47_1_gfXG` keep their names. Vapour pressure deficit
is now `VPD_T1_47_1`, gap-filled and complete, beside `VPD_T1_47_1_HOMOGENIZED`, and the
column `VPD_T1_47_1_gfXG` no longer exists. The raw, unfilled `TA_T1_47_1`, `SW_IN_T1_47_1` and
`PPFD_IN_T1_47_1` are no longer carried.

## ⚙️ Settings

In [1]:
# --- Sources -----------------------------------------------------------------------------------
# Both inputs live in the external (untracked) data folder and are addressed by absolute path, so
# the notebook does not depend on the working directory.
DATADIR = (r"F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data"
           r"\workflow")
FLUXFILE = rf"{DATADIR}\20_L1_FLUXES\IRGA72\99_IRGA72_L1_MERGED_2016-2026.09.parquet"
METEOFILE = rf"{DATADIR}\10_METEO\30_PRODUCTS\99_METEO_MERGED_2004-2026.09.parquet"

# --- Period ------------------------------------------------------------------------------------
# TIMESTAMP_MIDDLE of the first and the last record. The record starts with the calendar year, before
# the first flux record of 11 January 2016, so that 2016 carries meteo from its first half-hour.
# Both inputs must end at PERIOD_END, because both file names claim it.
PERIOD_START = '2016-01-01 00:15:00'
PERIOD_END = '2026-09-30 23:45:00'

# --- Output ------------------------------------------------------------------------------------
OUTNAME = "00_FPPC_IRGA72_INPUT_2016-2026.09"
OUTPATH = rf"{DATADIR}\30_FLUX_PROCESSING_CHAIN\IRGA72"

# --- Checks ------------------------------------------------------------------------------------
REQUIRED_TIME_RESOLUTION = '30min'
METEO_SUFFIX = '_T1_47_1'  # every meteo column name contains it, and no EddyPro column name does
# The drivers of the Level-4.1 gap-filling. They must be complete over the whole joined period.
DRIVERS = ['TA_T1_47_1_gfXG', 'SW_IN_T1_47_1_gfXG', 'VPD_T1_47_1']
FLUXCOLS = ['FC', 'LE', 'H']

In [2]:
from datetime import datetime
from pathlib import Path
import importlib.metadata
import textwrap

import pandas as pd

from diive.core.io.files import save_parquet, load_parquet

pd.set_option('display.max_rows', 60)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 1000)


def tabcap(text, width=100):
    """Print a table caption, above the table it belongs to."""
    print(textwrap.fill(f"Table - {text}", width) + "\n")


# Start time is kept so the last cell can report the total runtime of the notebook.
NOTEBOOK_START = datetime.now()
print(f"Last run: {NOTEBOOK_START.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"diive version: v{importlib.metadata.version('diive')}")

Last run: 2026-10-26 17:09:32
diive version: v0.93.0


## 📂 Load

In [3]:
flux = load_parquet(filepath=FLUXFILE)
meteo = load_parquet(filepath=METEOFILE)

for _name, _df in [('fluxes', flux), ('meteo', meteo)]:
    print(f"{_name:<7} {len(_df):>8,} records x {_df.shape[1]:>3} columns  "
          f"{_df.index[0]} -> {_df.index[-1]}  ({_df.index.name})")
    assert _df.index.name == 'TIMESTAMP_MIDDLE', f"{_name} is not indexed on TIMESTAMP_MIDDLE"
    assert _df.index[-1] == pd.Timestamp(PERIOD_END), f"{_name} does not end at PERIOD_END"
    assert not _df.index.duplicated().any(), f"{_name} has duplicate timestamps"
    _steps = _df.index.to_series().diff().dropna().unique()
    assert len(_steps) == 1 and _steps[0] == pd.Timedelta(REQUIRED_TIME_RESOLUTION), (
        f"{_name} is not on a continuous {REQUIRED_TIME_RESOLUTION} index")

> Loaded .parquet file 
F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\20_L1_FLUXES\IRGA
72\99_IRGA72_L1_MERGED_2016-2026.09.parquet (7.079 seconds).

> Loaded .parquet file 
F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\10_METEO\30_PRODU
CTS\99_METEO_MERGED_2004-2026.09.parquet (0.261 seconds).

fluxes   187,937 records x 329 columns  2016-01-11 15:45:00 -> 2026-09-30 23:45:00  (TIMESTAMP_MIDDLE)
meteo    398,832 records x  19 columns  2004-01-01 00:15:00 -> 2026-09-30 23:45:00  (TIMESTAMP_MIDDLE)


## 🔗 Join

In [4]:
# The two column sets must not share a name, or one would silently replace the other.
assert all(METEO_SUFFIX in _c for _c in meteo.columns), "a meteo column lacks the meteo suffix"
assert not any(METEO_SUFFIX in _c for _c in flux.columns), "a flux column carries the meteo suffix"
assert not set(flux.columns) & set(meteo.columns), "the two inputs share a column name"

# The joined period runs from PERIOD_START to PERIOD_END on the meteo index, which starts in 2004.
# The flux record must lie inside it; before its first record the flux columns are empty.
_start, _end = pd.Timestamp(PERIOD_START), pd.Timestamp(PERIOD_END)
assert meteo.index[0] <= _start, "the meteo record starts after PERIOD_START"
assert flux.index[0] >= _start, "the flux record starts before PERIOD_START"
df = meteo.loc[_start:_end].join(flux, how='left')
df = df.reindex(sorted(df.columns), axis=1)

print(f"Joined: {len(df):,} records x {df.shape[1]} columns "
      f"({flux.shape[1]} flux, {meteo.shape[1]} meteo), {df.index[0]} -> {df.index[-1]}")

Joined: 188,448 records x 348 columns (329 flux, 19 meteo), 2016-01-01 00:15:00 -> 2026-09-30 23:45:00


### 📋 Table: the meteo columns over the joined period

In [5]:
_rows = []
for _c in meteo.columns:
    _s = df[_c]
    _rows.append({'column': _c, 'present_%': round(100 * _s.notna().mean(), 2),
                  'first': _s.first_valid_index(), 'last': _s.last_valid_index()})
tabcap("every meteo column of the joined file, the share of the joined period it holds a value for, "
       "and its first and last value. PA and LW_IN keep their outages, because neither is gap-filled")
display(pd.DataFrame(_rows).set_index('column'))

Table - every meteo column of the joined file, the share of the joined period it holds a value for,
and its first and last value. PA and LW_IN keep their outages, because neither is gap-filled



,present_%,first,last
column,,,
SW_IN_T1_47_1_gfXG,100.00,2016-01-01 00:15:00,2026-09-30 23:45:00
FLAG_SW_IN_T1_47_1_ISFILLED,100.00,2016-01-01 00:15:00,2026-09-30 23:45:00
TA_T1_47_1_gfXG,100.00,2016-01-01 00:15:00,2026-09-30 23:45:00
FLAG_TA_T1_47_1_ISFILLED,100.00,2016-01-01 00:15:00,2026-09-30 23:45:00
TA_T1_47_1_HOMOGENIZED_gfXG,100.00,2016-01-01 00:15:00,2026-09-30 23:45:00
FLAG_TA_T1_47_1_SOURCE,100.00,2016-01-01 00:15:00,2026-09-30 23:45:00
PPFD_IN_T1_47_1_gfXG,100.00,2016-01-01 00:15:00,2026-09-30 23:45:00
FLAG_PPFD_IN_T1_47_1_ISFILLED,100.00,2016-01-01 00:15:00,2026-09-30 23:45:00
RH_T1_47_1,100.00,2016-01-01 00:15:00,2026-09-30 23:45:00


## 🔍 Checks

In [6]:
_period = meteo.loc[pd.Timestamp(PERIOD_START):pd.Timestamp(PERIOD_END)].index
assert df.index.equals(_period), "the joined index is not PERIOD_START to PERIOD_END"
assert df.index[0] == pd.Timestamp(PERIOD_START) and df.index[-1] == pd.Timestamp(PERIOD_END)
assert len(df.columns) == flux.shape[1] + meteo.shape[1], "a column was lost or duplicated"

# Nothing was changed on the way: every column equals its source.
# Before the first flux record every flux column is empty.
_before = df.index < flux.index[0]
assert df.loc[_before, flux.columns].isna().all().all(), "flux values before the flux record"
for _c in flux.columns:
    assert df.loc[flux.index, _c].equals(flux[_c]), f"flux column {_c} changed in the join"
for _c in meteo.columns:
    assert df[_c].equals(meteo.loc[_period, _c]), f"meteo column {_c} changed in the join"
print(f"{int(_before.sum()):,} records before the first flux record ({flux.index[0]}) carry meteo only")

# The gap-filling drivers must hold a value at every timestamp.
for _c in DRIVERS:
    _n = int(df[_c].isna().sum())
    assert _n == 0, f"driver {_c} has {_n} missing values over the joined period"
print(f"Index, columns and values verified. Drivers complete: {', '.join(DRIVERS)}")

511 records before the first flux record (2016-01-11 15:45:00) carry meteo only
Index, columns and values verified. Drivers complete: TA_T1_47_1_gfXG, SW_IN_T1_47_1_gfXG, VPD_T1_47_1


## 💾 Export

In [7]:
Path(OUTPATH).mkdir(parents=True, exist_ok=True)
filepath = save_parquet(filename=OUTNAME, data=df, outpath=OUTPATH)
print(f"Saved to: {filepath}")

> Saved file 
F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\30_FLUX_PROCESSIN
G_CHAIN\IRGA72\00_FPPC_IRGA72_INPUT_2016-2026.09.parquet (10.728 seconds).

Saved to: F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\30_FLUX_PROCESSING_CHAIN\IRGA72\00_FPPC_IRGA72_INPUT_2016-2026.09.parquet


## ✅ Read the written file back

In [8]:
_written = load_parquet(filepath=str(Path(OUTPATH) / f"{OUTNAME}.parquet"))

print(f"index:   {_written.index[0]} -> {_written.index[-1]}  ({len(_written):,} records)")
print(f"columns: {len(_written.columns):,}")

assert _written.index.equals(df.index), "index changed on write"
assert list(_written.columns) == list(df.columns), "columns changed on write"
for _c in FLUXCOLS + DRIVERS:
    assert _written[_c].equals(df[_c]), f"{_c} changed on write"
print("\nWritten file verified.")

> Loaded .parquet file 
F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\30_FLUX_PROCESSIN
G_CHAIN\IRGA72\00_FPPC_IRGA72_INPUT_2016-2026.09.parquet (0.466 seconds).

index:   2016-01-01 00:15:00 -> 2026-09-30 23:45:00  (188,448 records)
columns: 348

Written file verified.


***

# End of notebook.

In [9]:
NOTEBOOK_END = datetime.now()
_h, _rem = divmod(int((NOTEBOOK_END - NOTEBOOK_START).total_seconds()), 3600)
_m, _s = divmod(_rem, 60)

print(f"Started:  {NOTEBOOK_START.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Finished: {NOTEBOOK_END.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Runtime:  {_h:02d}:{_m:02d}:{_s:02d} (hh:mm:ss)")

Started:  2026-10-26 17:09:32
Finished: 2026-10-26 17:10:09
Runtime:  00:00:36 (hh:mm:ss)
